# Stage 2: Dissatisfaction Classifier Training (mBERT, multi-label)

Trains on SentiTaglish_ProductsAndServices_Dissatisfaction_Classification.csv
to classify WHY a negative review is negative, across 13 categories:
def, dam, perf, pmq, lst, var, mi, auth, pss, del, pack, val

Unlike Stage 1 (one sentiment per review), a review can belong to MULTIPLE
dissatisfaction categories at once (72% of the labeled data has 2+ labels).
This means: different loss function (binary cross-entropy per label, not
softmax over one label), different metrics (per-label F1 + micro/macro
averages, not simple accuracy), and a different split strategy (iterative
stratification for multi-label, not sklearn's standard stratify).

**Before running:** set `SMOKE_TEST = True` first, run everything once to
confirm no errors. Then set `SMOKE_TEST = False` and run the full thing.

**Also check:** DATA_PATH matches wherever your uploaded dataset landed
under /kaggle/input/. Use the copy-path icon in the Data panel to get the
exact path rather than typing it by hand.

## 1. Setup

In [1]:
!pip install -q -U transformers datasets scikit-learn
!pip install -q iterative-stratification

In [2]:
import re
import unicodedata
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    HAS_ITERSTRAT = True
except ImportError:
    HAS_ITERSTRAT = False
    print("iterative-stratification not available, will fall back to random split")

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5060


In [3]:
import transformers
import torch
import sklearn
import datasets as hf_datasets

try:
    import iterstrat
    iterstrat_version = getattr(iterstrat, "__version__", "unknown (no __version__ attr)")
except ImportError:
    iterstrat_version = "not installed"

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("sklearn:", sklearn.__version__)
print("datasets:", hf_datasets.__version__)
print("iterative-stratification:", iterstrat_version)

# Also worth checking what attention backend the model actually loaded with
from transformers import AutoModelForSequenceClassification
model_check = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base", num_labels=12, problem_type="multi_label_classification"
)
print("Attention implementation:", model_check.config._attn_implementation)

transformers: 5.18.0
torch: 2.14.0+cu130
sklearn: 1.9.1
datasets: 5.0.1
iterative-stratification: 0.1.9


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Attention implementation: sdpa


## 2. Config

In [4]:
# --- CHECK THIS PATH ---
# Use the copy-path icon in Kaggle's Data panel to get the exact path.
DATA_PATH = "../data/SentiTaglish_ProductsAndServices_Dissatisfaction_Classification.csv"
OUTPUT_DIR = "../models/stage2_dissatisfaction"


SMOKE_TEST = False
SMOKE_TEST_SIZE = 200
EPOCHS = 10

SEED = 42
MODEL_NAME = "bert-base-multilingual-uncased"
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
BATCH_SIZE = 8
PREDICTION_THRESHOLD = 0.5   # sigmoid output above this counts as a positive label

CATEGORY_COLUMNS = ["def", "dam", "perf", "pmq", "lst", "var", "mi",
                    "auth", "pss", "del", "pack", "val"]
NUM_LABELS = len(CATEGORY_COLUMNS)
ID2LABEL = {i: name for i, name in enumerate(CATEGORY_COLUMNS)}
LABEL2ID = {name: i for i, name in enumerate(CATEGORY_COLUMNS)}

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. Check Notebook options > Accelerator is set to GPU.")

Using device: cuda


## 3. Text cleaning

In [5]:
SPELLING_MAP = {
    "d2": "dito", "dto": "dito",
    "un": "yun",
    "sya": "siya", "cya": "siya",
    "nde": "hindi", "hnd": "hindi",
    "wla": "wala", "wlang": "walang",
    "eto": "ito",
    "pde": "pwede", "pwd": "pwede",
    "gud": "good",
    "thnx": "thanks", "tnx": "thanks",
    "salamt": "salamat",
}

_REPEATED_CHAR = re.compile(r"(.)\1{2,}")
_WHITESPACE = re.compile(r"\s+")

def clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = _REPEATED_CHAR.sub(r"\1\1", text)
    tokens = text.split()
    tokens = [SPELLING_MAP.get(t, t) for t in tokens]
    text = " ".join(tokens)
    text = _WHITESPACE.sub(" ", text).strip()
    return text

print(clean_text("SOBRAAAA panget ng quality, sira agad!!"))

sobraa panget ng quality, sira agad!!


## 4. Load and prepare data

In [6]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows")

# Handle either 'review' or 'a' as the text column name (spreadsheet exports
# sometimes rename the first column oddly)
if "review" not in df.columns and "a" in df.columns:
    df = df.rename(columns={"a": "review"})

missing_cols = [c for c in CATEGORY_COLUMNS if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing expected category columns: {missing_cols}. Found: {list(df.columns)}")

df["review_clean"] = df["review"].apply(clean_text)
df = df[df["review_clean"] != ""].reset_index(drop=True)

labels_matrix = df[CATEGORY_COLUMNS].values.astype(float)
print(f"\nAfter cleaning: {len(df)} rows")

print("\nPer-category positive counts:")
for i, cat in enumerate(CATEGORY_COLUMNS):
    count = int(labels_matrix[:, i].sum())
    print(f"  {cat:6s}: {count:5d} ({100*count/len(df):.1f}%)")

label_counts_per_row = labels_matrix.sum(axis=1)
print(f"\nRows with 2+ labels: {(label_counts_per_row >= 2).sum()} ({100*(label_counts_per_row>=2).sum()/len(df):.1f}%)")

if SMOKE_TEST:
    df = df.sample(min(len(df), SMOKE_TEST_SIZE), random_state=SEED).reset_index(drop=True)
    labels_matrix = df[CATEGORY_COLUMNS].values.astype(float)
    epochs = 1
    print(f"\n[SMOKE TEST] Using {len(df)} rows, 1 epoch")
else:
    epochs = EPOCHS

# Multi-label stratified split: ensures rare categories (e.g. auth at ~2%)
# appear in both train and val, unlike a plain random split.
if HAS_ITERSTRAT and len(df) > 20:
    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_idx, val_idx = next(msss.split(df["review_clean"], labels_matrix))
else:
    from sklearn.model_selection import train_test_split
    train_idx, val_idx = train_test_split(
        np.arange(len(df)), test_size=0.2, random_state=SEED
    )
    print("Using random split (iterstrat unavailable or dataset too small for smoke test)")

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
train_labels = labels_matrix[train_idx]
val_labels = labels_matrix[val_idx]

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)}")

Loaded 3408 rows

After cleaning: 3408 rows

Per-category positive counts:
  def   :   710 (20.8%)
  dam   :   502 (14.7%)
  perf  :   392 (11.5%)
  pmq   :   742 (21.8%)
  lst   :   822 (24.1%)
  var   :   878 (25.8%)
  mi    :   379 (11.1%)
  auth  :    61 (1.8%)
  pss   :  1268 (37.2%)
  del   :   224 (6.6%)
  pack  :   251 (7.4%)
  val   :   474 (13.9%)

Rows with 2+ labels: 2460 (72.2%)

Train: 2708 | Val: 700


## 5. Tokenize, Train, and Evaluate (5-fold CV)

In [7]:
import gc, os
import numpy as np, pandas as pd, torch, torch.nn as nn
from datasets import Dataset
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss, accuracy_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          set_seed)                                   # EDIT 1: added set_seed

N_FOLDS = 5
EPOCHS = 10
MODEL_NAME = "bert-base-multilingual-uncased"   # rerun with "bert-base-multilingual-uncased"
TAG = MODEL_NAME.split("/")[-1]
MAX_POS_WEIGHT = 10.0
os.makedirs("../results", exist_ok=True)

Y = df[CATEGORY_COLUMNS].values.astype(float)

class WeightedBCETrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)(outputs.logits, labels.float())
        return (loss, outputs) if return_outputs else loss

def make_ds(tok, texts, labels):
    enc = tok(list(texts), truncation=True, max_length=MAX_LENGTH)
    enc["labels"] = labels.tolist()
    return Dataset.from_dict(enc)

mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
rows = []

for fold, (tr, va) in enumerate(mskf.split(np.zeros(len(df)), Y)):
    set_seed(SEED)                                                    # EDIT 2: first line in the loop
    print(f"\n=== Fold {fold + 1}/{N_FOLDS} ===")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, problem_type="multi_label_classification",
        id2label=ID2LABEL, label2id=LABEL2ID,
    )
    train_ds = make_ds(tokenizer, df["review_clean"].iloc[tr], Y[tr])
    val_ds = make_ds(tokenizer, df["review_clean"].iloc[va], Y[va])

    num_pos = Y[tr].sum(axis=0)
    pw = np.clip((len(tr) - num_pos) / np.clip(num_pos, 1, None), None, MAX_POS_WEIGHT)
    pos_weight = torch.tensor(pw, dtype=torch.float32, device=device)

    total_steps = int(np.ceil(len(tr) / BATCH_SIZE)) * EPOCHS         # EDIT 3a: right before TrainingArguments
    args = TrainingArguments(
        output_dir=f"{OUTPUT_DIR}/{TAG}_fold{fold}",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=32,
        num_train_epochs=EPOCHS,
        warmup_steps=int(0.1 * total_steps),                          # EDIT 3b
        weight_decay=0.01,                                            # EDIT 3c
        save_strategy="no",
        seed=SEED,
        report_to="none",
    )
    trainer = WeightedBCETrainer(model=model, args=args, train_dataset=train_ds,
                                 data_collator=DataCollatorWithPadding(tokenizer),
                                 pos_weight=pos_weight)
    trainer.train()

    out = trainer.predict(val_ds)
    probs = 1 / (1 + np.exp(-out.predictions))
    p = (probs >= PREDICTION_THRESHOLD).astype(int)
    y = out.label_ids.astype(int)

    row = {
        "model": TAG, "fold": fold,
        "macro_f1": f1_score(y, p, average="macro", zero_division=0),
        "micro_f1": f1_score(y, p, average="micro", zero_division=0),
        "macro_precision": precision_score(y, p, average="macro", zero_division=0),
        "micro_precision": precision_score(y, p, average="micro", zero_division=0),
        "macro_recall": recall_score(y, p, average="macro", zero_division=0),
        "micro_recall": recall_score(y, p, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(y, p),
        "subset_accuracy": accuracy_score(y, p),
    }
    for i, cat in enumerate(CATEGORY_COLUMNS):
        row[f"f1_{cat}"] = f1_score(y[:, i], p[:, i], zero_division=0)
    rows.append(row)
    pd.DataFrame(rows).to_csv(f"../results/stage2_{TAG}_folds_partial.csv", index=False)

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

res = pd.DataFrame(rows)
res.to_csv(f"../results/stage2_{TAG}_folds.csv", index=False)
print(res.drop(columns=["model", "fold"]).describe().loc[["mean", "std"]].round(4).T)


=== Fold 1/5 ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.979681
1000,0.662544
1500,0.455117
2000,0.335749
2500,0.246167
3000,0.193102



=== Fold 2/5 ===


...


=== Fold 5/5 ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.984989
1000,0.665568
1500,0.459361
2000,0.328459
2500,0.243257
3000,0.189254


                   mean     std
macro_f1         0.7199  0.0100
micro_f1         0.7455  0.0048
macro_precision  0.6843  0.0128
micro_precision  0.7009  0.0071
macro_recall     0.7639  0.0199
micro_recall     0.7964  0.0112
hamming_loss     0.0891  0.0020
subset_accuracy  0.3674  0.0033
f1_def           0.7535  0.0063
f1_dam           0.7229  0.0291
f1_perf          0.6363  0.0350
f1_pmq           0.6888  0.0281
f1_lst           0.6905  0.0278
f1_var           0.8586  0.0142
f1_mi            0.7333  0.0437
f1_auth          0.5714  0.0958
f1_pss           0.7870  0.0140
f1_del           0.6929  0.0679
f1_pack          0.7273  0.0384
f1_val           0.7761  0.0396
